# 配套实践 07-02：观测与动作数据契约

本练习构造一个最小机器人 batch，检查本体、力、动作和时间 mask 的形状，并完成标准化、反归一化、限幅和接触保护。依赖：NumPy；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/basics/07-robot-state-and-action/" target="_blank">返回课程正文</a>

In [ ]:
import numpy as np  # 构造时序 batch 并执行尺度变换
rng = np.random.default_rng(21)  # 创建固定种子的随机数生成器

## 1. 用 schema 明确每个维度

示例使用六维本体状态、六维力/力矩和七维末端增量动作。真实项目应把字段名、单位、坐标系、频率和版本一并保存。

In [ ]:
batch_size, time_steps = 3, 5  # 定义三个样本和五个时间步
schema = {"version": "1.0", "frequency_hz": 20.0, "proprio_fields": ["q1_rad", "q2_rad", "dq1_rad_s", "dq2_rad_s", "gripper_m", "contact_prob"], "wrench_fields": ["fx_N", "fy_N", "fz_N", "mx_Nm", "my_Nm", "mz_Nm"], "action_fields": ["dx_m", "dy_m", "dz_m", "drx_rad", "dry_rad", "drz_rad", "gripper"]}  # 用字段名和单位定义数据契约
proprio = rng.normal(size=(batch_size, time_steps, 6))  # 生成示例本体时序 [B,T,P]
wrench = rng.normal(scale=0.4, size=(batch_size, time_steps, 6))  # 生成示例力和力矩时序 [B,T,F]
valid_mask = np.array([[1, 1, 1, 1, 1], [1, 1, 1, 0, 0], [1, 1, 1, 1, 0]], dtype=bool)  # 标出每条轨迹的真实时间步
assert proprio.shape[:2] == valid_mask.shape  # 检查本体时间维与有效位 mask 对齐
assert wrench.shape[:2] == valid_mask.shape  # 检查力信号时间维与有效位 mask 对齐
assert proprio.shape[-1] == len(schema["proprio_fields"])  # 检查本体维度与字段清单一致
print("schema version:", schema["version"])  # 输出接口版本号
print("proprio / wrench / mask:", proprio.shape, wrench.shape, valid_mask.shape)  # 显示三类张量形状

**怎样理解上一结果：** 三个张量的前两维都对应 `[B,T]`，因此同一个 `valid_mask` 可以标出每条轨迹中的真实时间步。形状一致只通过了最基本检查，字段顺序、单位、频率和 schema 版本仍必须随数据保存。

## 2. 只用有效训练数据计算统计量

padding 时间步不能参与均值和标准差。归一化参数属于模型版本的一部分，推理时必须使用训练阶段保存的同一组数值。

In [ ]:
valid_proprio = proprio[valid_mask]  # 使用布尔 mask 取出全部真实时间步
mean = valid_proprio.mean(axis=0)  # 按本体维度计算训练均值
std = valid_proprio.std(axis=0) + 1e-6  # 按本体维度计算标准差并避免除零
normalized = (proprio - mean) / std  # 将本体状态转换到模型使用的标准化空间
restored = normalized * std + mean  # 使用相同统计量恢复原始物理空间
round_trip_error = np.max(np.abs(restored[valid_mask] - proprio[valid_mask]))  # 检查真实时间步的往返误差
print(f"归一化往返最大误差: {round_trip_error:.3e}")  # 输出数值恢复精度
assert round_trip_error < 1e-10  # 确认归一化与反归一化互为逆操作

**怎样理解上一结果：** 往返误差接近机器浮点精度，说明使用同一组均值和标准差可以恢复有效时间步。这个检查不能允许测试数据参与统计量计算，也不能代替对 padding、异常值和传感器漂移的检查。

## 3. 从模型动作恢复物理动作并限幅

前三维是基座坐标中的米制位置增量，中间三维是弧度旋转增量，最后一维是归一化夹爪命令。不同维度使用不同物理上下界。

In [ ]:
action_low = np.array([-0.02, -0.02, -0.02, -0.08, -0.08, -0.08, -1.0])  # 定义每步动作的物理下界
action_high = np.array([0.02, 0.02, 0.02, 0.08, 0.08, 0.08, 1.0])  # 定义每步动作的物理上界
model_action = np.array([[1.35, -0.50, 0.25, 0.20, -1.40, 0.10, 0.80], [0.40, 0.20, -0.10, 0.30, 0.10, 0.00, -0.70]])  # 模拟两个可能超出标准范围的模型输出
clipped_model_action = np.clip(model_action, -1.0, 1.0)  # 先把模型空间动作限制在约定范围
physical_action = action_low + (clipped_model_action + 1.0) * 0.5 * (action_high - action_low)  # 将标准化动作线性恢复到物理单位
print("恢复后的物理动作：\n", physical_action.round(4))  # 显示米、弧度和夹爪命令
assert np.all(physical_action >= action_low) and np.all(physical_action <= action_high)  # 验证全部动作满足接口边界

**怎样理解上一结果：** 第一条模型输出含有超过标准范围的分量，先裁剪再反归一化后，三维平移没有超过每步 2 厘米，旋转没有超过 0.08 弧度。输出数组混合米、弧度和无量纲夹爪值，因此不能只看七维形状就认为动作含义明确。

## 4. 接触触发的最小安全修正

下面只演示接口思想：法向接触力过大时缩小平移动作。真实机器人必须使用经过验证的安全控制器、碰撞检测和急停，不能依赖这段教学规则。

In [ ]:
measured_force_z = np.array([2.5, 14.0])  # 模拟两次执行前测得的法向接触力并使用牛顿
force_threshold = 8.0  # 设置教学用接触力阈值
safe_action = physical_action.copy()  # 复制动作避免覆盖原始策略输出
contact_rows = measured_force_z > force_threshold  # 找到接触力超过阈值的样本
safe_action[contact_rows, :3] *= 0.15  # 对高接触样本显著缩小三维平移增量
print("高接触修正前平移：", physical_action[1, :3].round(4))  # 显示修正前第二条平移动作
print("高接触修正后平移：", safe_action[1, :3].round(4))  # 显示修正后第二条平移动作
assert np.linalg.norm(safe_action[1, :3]) < np.linalg.norm(physical_action[1, :3])  # 确认接触保护缩小了动作幅度

**怎样理解结果：** 第二个样本的法向力超过教学阈值后，平移向量按比例缩小，但旋转和夹爪命令没有被改动。这只是为了显示安全层位于模型输出与控制器之间；固定阈值和缩放比例没有经过硬件验证，不能直接用于真实机器人。

## 结论与练习

可靠动作接口要同时保存字段顺序、单位、坐标系、控制频率、统计量和上下界。尝试把控制频率从 20 Hz 改为 50 Hz：若模型输出的是“每步增量”，应该怎样缩放才能保持相同米制速度？